# Preprocessing — Credit Card Transaction Fraud

Builds the modelling-ready feature files from the raw Sparkov data.

Input (gitignored, see `eda.ipynb`): `data/fraudTrain.csv`, `data/fraudTest.csv`, and `data/kyc_customers.csv` (run `kyc.ipynb` first)

Output: `data/processed/fraudTrain_features.csv`, `data/processed/fraudTest_features.csv`

This notebook covers:
1. Loading data & dropping rows with missing gender
2. Train-only thresholds
3. Time features (incl. cyclical encodings)
4. Amount and age features
5. Distance features
6. KYC features (synthetic customer profile)
7. Behavioural features (velocity and deviation from the customer's own history)
8. Merchant category
9. Dropping identifiers and redundant columns
10. One-hot encoding (avoiding the dummy variable trap)
11. Multicollinearity check
12. Saving the output

All transformations are target-independent: `is_fraud` is never used to build a feature, including the KYC and behavioural ones. It is carried through unchanged in both output files as the label.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "processed"

## 1. Load data

`gender`, `category` and `state` have no missing values in either file. Rows with a missing `gender` are still dropped as a safeguard: with `drop_first=True` encoding (section 10) there is no missing-value indicator, so a missing gender would otherwise be silently encoded as the baseline (`F`).

In [2]:
train = pd.read_csv(DATA_DIR / "fraudTrain.csv", index_col=0)
test = pd.read_csv(DATA_DIR / "fraudTest.csv", index_col=0)

for name, df in [("train", train), ("test", test)]:
    print(f"{name}: {df.shape}, missing gender: {df['gender'].isna().sum():,}")

train = train.dropna(subset=["gender"])
test = test.dropna(subset=["gender"])

train: (1296675, 22), missing gender: 0
test: (555719, 22), missing gender: 0


## 2. Train-only thresholds

`amt_is_large` and `distance_is_large` flag the top 5% of transactions. The 95th-percentile cut-offs are computed on the **training set only** and applied unchanged to the test set, so the test distribution never influences preprocessing.

In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two latitude/longitude pairs."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * 6371 * np.arcsin(np.sqrt(a))


amount_95 = train["amt"].quantile(0.95)
distance_95 = haversine_km(
    train["lat"], train["long"], train["merch_lat"], train["merch_long"]
).quantile(0.95)

print(f"95th-percentile amount:   {amount_95:.2f}")
print(f"95th-percentile distance: {distance_95:.2f} km")

95th-percentile amount:   196.31
95th-percentile distance: 120.50 km


## 3. Time features

The raw timestamp is split into `month`, `day_of_month`, `hour` and `day_of_week` (Monday = 0), plus two flags: `is_weekend` and `is_night` (00:00–05:59).

**No `year` feature:** it has only two values (2019, 2020) and the test set is entirely 2020, so the model cannot learn anything about year that transfers to the test period.

**Cyclical encodings.** Hour, day of week and month wrap around: 23:00 is next to 00:00, Sunday next to Monday, December next to January. As raw integers these neighbours look far apart (23 vs 0). Mapping each value to a point on a circle fixes this:

```
angle = 2π × value / period      (period = 24, 7 or 12)
x_sin = sin(angle),  x_cos = cos(angle)
```

Both are needed: `sin` alone gives the same value for mirror-image points (e.g. 02:00 and 10:00), but the `(sin, cos)` pair is unique for every value.

In [4]:
def add_time_features(df):
    df = df.copy()
    timestamp = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")

    df["month"] = timestamp.dt.month
    df["day_of_month"] = timestamp.dt.day
    df["hour"] = timestamp.dt.hour
    df["day_of_week"] = timestamp.dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype("int8")
    df["is_night"] = df["hour"].isin([0, 1, 2, 3, 4, 5]).astype("int8")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

## 4. Amount and age features

- `log_amt` = `log1p(amt)`, reducing the heavy right skew of `amt`
- `amt_is_large` = 1 when `amt` ≥ the training 95th percentile
- `age` = cardholder age at transaction time, from `dob`

In [5]:
def add_amount_age_features(df, amount_95):
    df = df.copy()
    timestamp = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    dob = pd.to_datetime(df["dob"], errors="coerce")

    df["age"] = ((timestamp - dob).dt.days / 365.25).round().astype("float")
    df["log_amt"] = np.log1p(df["amt"].clip(lower=0))
    df["amt_is_large"] = (df["amt"] >= amount_95).astype("int8")
    return df

## 5. Distance features

Distance between cardholder and merchant (Haversine) is kept only as `log_distance_km` and `distance_is_large`.

Raw `distance_km` and the earlier interaction features (`online_far_transaction` = online × distance, `amount_distance_interaction` = amount × distance) are **not** kept:
- they had VIFs of 11–81, being built from `log_distance_km` and `log_amt`;
- distance shows no relationship with fraud in this dataset (table below). The simulator places every merchant within ~150 km of the cardholder, so no transaction is genuinely "far away".

In [6]:
check = train[["category", "is_fraud"]].copy()
check["distance_km"] = haversine_km(
    train["lat"], train["long"], train["merch_lat"], train["merch_long"]
)
check["channel"] = np.select(
    [check["category"].str.endswith("_net"), check["category"].str.endswith("_pos")],
    ["online", "pos"],
    default="other",
)
check["distance_quintile"] = pd.qcut(
    check["distance_km"], 5, labels=["q1 near", "q2", "q3", "q4", "q5 far"]
)

# Fraud rate (%) is flat across distance for every channel
(
    check.pivot_table(
        index="distance_quintile", columns="channel", values="is_fraud",
        aggfunc="mean", observed=True,
    ) * 100
).round(2)

channel,online,other,pos
distance_quintile,,,
q1 near,1.36,0.23,0.85
q2,1.35,0.27,0.91
q3,1.31,0.24,0.91
q4,1.30,0.25,0.89
q5 far,1.38,0.25,0.86


In [7]:
def add_distance_features(df, distance_95):
    df = df.copy()
    distance_km = haversine_km(df["lat"], df["long"], df["merch_lat"], df["merch_long"])
    df["log_distance_km"] = np.log1p(distance_km.clip(lower=0))
    df["distance_is_large"] = (distance_km >= distance_95).astype("int8")
    return df

## 6. KYC features

The synthetic KYC table from `kyc.ipynb` (one row per customer) is joined to every transaction on `cc_num`. It was generated without fraud labels. All KYC columns are prefixed `kyc_`, so the model notebook can compare feature sets.

**Customer profile:**

| Feature | From the KYC table |
|---|---|
| `kyc_log_annual_income` | log of declared annual income |
| `kyc_credit_score` | credit score |
| `kyc_declared_txn_per_month` | expected transactions per month |
| `kyc_num_products` | products held with the bank |
| `kyc_account_tenure_years` | years since the account was opened, **at the time of each transaction** |
| `kyc_address_verified`, `kyc_phone_verified`, `kyc_email_verified` | 0/1 verification flags |
| `kyc_pep_flag`, `kyc_sanctions_potential_match`, `kyc_review_overdue` | 0/1 screening and review flags |
| `kyc_risk_rating` | 0 = Low, 1 = Medium, 2 = High (ordered, so kept as one number) |
| `kyc_employment_status`, `kyc_residency_status`, `kyc_id_verification_method` | one-hot encoded in section 10 |

**Profile deviation**, i.e. how this transaction compares with what the customer declared. This is how transaction monitoring uses KYC in practice:

| Feature | Definition |
|---|---|
| `kyc_amt_to_declared_spend` | `amt` ÷ declared monthly spend (0.5 = this one purchase is half a month's expected spend) |
| `kyc_amt_to_credit_limit` | `amt` ÷ credit limit |

**Left out on purpose:**
- `onboarding_channel`: "Branch" always means "Document (in person)" verification, so the channel dummies would be exactly collinear with the ID-method dummies.
- `source_of_funds`: almost fully determined by employment status.
- `income_band`: a binned copy of income.
- `declared_monthly_spend` and `credit_limit` levels: both are derived from income (the credit limit had a VIF of 12). Their information enters through the two ratios above.
- `kyc_risk_points`: a near-copy of the rating.

In [8]:
kyc = pd.read_csv(DATA_DIR / "kyc_customers.csv", parse_dates=["account_open_date"]).set_index("cc_num")

KYC_FLAGS = {
    "address_verified": "kyc_address_verified",
    "phone_verified": "kyc_phone_verified",
    "email_verified": "kyc_email_verified",
    "pep_flag": "kyc_pep_flag",
    "kyc_review_overdue": "kyc_review_overdue",
}
KYC_CATEGORICAL = ["employment_status", "residency_status", "id_verification_method"]


def add_kyc_features(df):
    df = df.copy()
    profile = kyc.loc[df["cc_num"]].set_axis(df.index)  # the customer's KYC row, repeated per transaction
    timestamp = pd.to_datetime(df["trans_date_trans_time"])

    df["kyc_log_annual_income"] = np.log(profile["annual_income"])
    df["kyc_credit_score"] = profile["credit_score"]
    df["kyc_declared_txn_per_month"] = profile["declared_txn_per_month"]
    df["kyc_num_products"] = profile["num_products"]
    df["kyc_account_tenure_years"] = (timestamp - profile["account_open_date"]).dt.days / 365.25
    for column, name in KYC_FLAGS.items():
        df[name] = profile[column].astype("int8")
    df["kyc_sanctions_potential_match"] = (profile["sanctions_screening"] != "Clear").astype("int8")
    df["kyc_risk_rating"] = profile["kyc_risk_rating"].map({"Low": 0, "Medium": 1, "High": 2})
    for column in KYC_CATEGORICAL:
        df[f"kyc_{column}"] = profile[column]

    df["kyc_amt_to_declared_spend"] = df["amt"] / profile["declared_monthly_spend"]
    df["kyc_amt_to_credit_limit"] = df["amt"] / profile["credit_limit"]
    return df


print(f"KYC table: {kyc.shape[0]:,} customers; all transaction customers covered:",
      set(train["cc_num"]).union(test["cc_num"]) <= set(kyc.index))

KYC table: 999 customers; all transaction customers covered: True


## 7. Behavioural features

These describe how the customer has been **behaving recently** compared with their own history. Fraud in this data arrives as a burst of about 10 transactions in a day, which is exactly what velocity features pick up. All columns are prefixed `beh_`.

| Feature | Definition |
|---|---|
| `beh_txn_count_1h` | the customer's transactions in the past hour, including this one |
| `beh_txn_count_24h` | the same over the past 24 hours |
| `beh_log_amt_sum_24h` | log of total spend over the past 24 hours, including this one |
| `beh_log_secs_since_prev` | log of seconds since the customer's previous transaction (capped at 30 days; first transaction = 30 days) |
| `beh_amt_to_cust_avg` | this amount ÷ the average of all the customer's **earlier** transactions (1 = typical, 10 = ten times usual; capped at 20, first transaction = 1) |

**No leakage:** each feature uses only the current transaction and the customer's earlier ones, never later transactions and never labels. Train and test are processed as one timeline, so the first test transactions can see the end of the training period, just as a live system would see its recent history.

In [9]:
def behavioural_features(train, test):
    columns = ["cc_num", "amt", "trans_date_trans_time"]
    combined = pd.concat([train[columns], test[columns]], keys=["train", "test"], names=["split", "row"])
    combined["time"] = pd.to_datetime(combined["trans_date_trans_time"])
    combined = combined.sort_values(["cc_num", "time"], kind="stable")

    by_customer = combined.groupby("cc_num", sort=True)
    rolling_amt = combined.set_index("time").groupby("cc_num", sort=True)["amt"]  # same row order as combined

    features = pd.DataFrame(index=combined.index)
    features["beh_txn_count_1h"] = rolling_amt.rolling("1h").count().to_numpy()
    features["beh_txn_count_24h"] = rolling_amt.rolling("24h").count().to_numpy()
    features["beh_log_amt_sum_24h"] = np.log1p(rolling_amt.rolling("24h").sum().to_numpy())

    max_gap = 30 * 24 * 3600
    seconds_since_prev = by_customer["time"].diff().dt.total_seconds().fillna(max_gap).clip(upper=max_gap)
    features["beh_log_secs_since_prev"] = np.log1p(seconds_since_prev)

    earlier_count = by_customer.cumcount()
    earlier_mean = (by_customer["amt"].cumsum() - combined["amt"]) / earlier_count.replace(0, np.nan)
    # A plain ratio rather than a log difference: log(amt) − log(average) would be a near-copy of log_amt (VIF ~35)
    features["beh_amt_to_cust_avg"] = (combined["amt"] / earlier_mean.fillna(combined["amt"])).clip(upper=20)

    return features.loc["train"].reindex(train.index), features.loc["test"].reindex(test.index)


beh_train, beh_test = behavioural_features(train, test)
train = train.join(beh_train)
test = test.join(beh_test)

# Fraud rate by recent activity (shown for illustration; labels are not used to build the features)
pd.crosstab(
    pd.cut(train["beh_txn_count_24h"], [0, 2, 5, 10, 20, np.inf], labels=["1–2", "3–5", "6–10", "11–20", "21+"]),
    train["is_fraud"], normalize="index",
).rename(columns={0: "legitimate", 1: "fraud"}).rename_axis(index="transactions in past 24h", columns=None).round(4)

,legitimate,fraud
transactions in past 24h,,
1–2,0.9968,0.0032
3–5,0.9940,0.0060
6–10,0.9919,0.0081
11–20,0.9974,0.0026
21+,0.9994,0.0006


## 8. Merchant category

`category` is kept and one-hot encoded (section 10), so models can tell apart e.g. `grocery_net` and `grocery_pos`.

`is_online`, `is_pos` and a suffix-free `category_base` are **not** created, because each is an exact sum of `category` dummies, which is perfect multicollinearity:
- `is_online` = `category_grocery_net` + `category_misc_net` + `category_shopping_net`
- `is_pos` = the sum of the three `_pos` dummies
- `category_base_shopping` = `category_shopping_net` + `category_shopping_pos`, and so on

Even `category_base` + `is_online` + `is_pos` together are collinear (`is_online + is_pos` = the grocery + misc + shopping base dummies), so the cleanest option is to keep `category` alone.

## 9. Drop identifiers and redundant columns

| Column(s) | Reason |
|---|---|
| `trans_date_trans_time`, `dob` | Replaced by time features and `age` |
| `trans_num`, `cc_num` | Identifiers; a card number has no numeric meaning (`cc_num` is used first, to join the KYC table and group the behavioural features) |
| `first`, `last`, `street` | Personal information |
| `city`, `zip`, `merchant`, `job` | High-cardinality; excluded from this baseline |
| `unix_time` | Same information as `month`/`day_of_month`/`hour` (VIF > 100,000) |
| `merch_lat`, `merch_long` | Near-copies of cardholder coordinates (VIF 78–570); their only distinct information is the distance |
| `lat`, `long` | Largely determined by `state` (VIF up to 650 with the state dummies); `state` is kept as the more interpretable location feature |

In [10]:
COLUMNS_TO_DROP = [
    "trans_date_trans_time", "dob", "trans_num", "cc_num",
    "first", "last", "street", "city", "zip", "merchant", "job",
    "unix_time", "merch_lat", "merch_long", "lat", "long",
]


def build_features(df):
    df = add_time_features(df)
    df = add_amount_age_features(df, amount_95)
    df = add_distance_features(df, distance_95)
    df = add_kyc_features(df)
    return df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns])


train_features = build_features(train)
test_features = build_features(test)
train_features.head()

,category,amt,gender,state,city_pop,is_fraud,beh_txn_count_1h,beh_txn_count_24h,beh_log_amt_sum_24h,beh_log_secs_since_prev,beh_amt_to_cust_avg,month,day_of_month,hour,day_of_week,is_weekend,is_night,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,age,log_amt,amt_is_large,log_distance_km,distance_is_large,kyc_log_annual_income,kyc_credit_score,kyc_declared_txn_per_month,kyc_num_products,kyc_account_tenure_years,kyc_address_verified,kyc_phone_verified,kyc_email_verified,kyc_pep_flag,kyc_review_overdue,kyc_sanctions_potential_match,kyc_risk_rating,kyc_employment_status,kyc_residency_status,kyc_id_verification_method,kyc_amt_to_declared_spend,kyc_amt_to_credit_limit
0,misc_net,4.97,F,NC,3495,0,1.0,1.0,1.786747,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,31.0,1.786747,0,4.376984,0,11.050890,591,41,3,7.498973,1,1,1,0,0,0,0,Employed,Citizen,Database check,0.005522,0.000710
1,grocery_pos,107.23,F,WA,149,0,1.0,1.0,4.684259,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,41.0,4.684259,0,3.440808,0,11.211820,703,72,1,1.815195,1,1,1,0,0,0,0,Employed,Citizen,Document + biometric,0.049874,0.007659
2,entertainment,220.11,M,ID,4154,0,1.0,1.0,5.398660,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,57.0,5.398660,1,4.693237,0,11.018629,557,29,5,3.857632,1,1,1,0,0,0,0,Employed,Citizen,Document (in person),0.209629,0.048913
3,gas_transport,45.00,M,MT,1939,0,1.0,1.0,3.828641,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,52.0,3.828641,0,4.571337,0,10.596635,682,33,1,3.411362,1,1,1,0,0,0,0,Employed,Permanent resident,Document upload,0.037500,0.004500
4,misc_pos,41.96,M,VA,99,0,1.0,1.0,3.760269,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,33.0,3.760269,0,4.363821,0,9.210340,590,41,2,7.466119,1,1,1,0,0,0,0,Self-employed,Citizen,Document + biometric,0.279733,0.041960


## 10. One-hot encoding

The remaining text columns (`category`, `gender`, `state`, and the KYC fields `kyc_employment_status`, `kyc_residency_status`, `kyc_id_verification_method`) become 0/1 indicator columns. Numbering categories instead (grocery = 0, shopping = 1, …) would invent an order and distances that don't exist.

**Dummy variable trap.** A full set of dummies always sums to 1 (e.g. `gender_F + gender_M = 1`), which is perfectly collinear with the intercept. `drop_first=True` drops one category per column as the **baseline**; each remaining coefficient is read relative to it.

The baseline is the **most frequent category in the training set**: `F` for gender, `gas_transport` for category, `TX` for state. A rare baseline causes near-collinearity: with `AK` (2,120 rows) as the baseline, the other 50 state dummies sum to 1 for 99.8% of rows, which pushes state VIFs up to ~40.

`is_fraud` is set aside before encoding and added back afterwards. Train and test are encoded against the same category list so both files end up with identical columns. Indicators are stored sparsely to save memory.

In [11]:
def one_hot_encode(train_features, test_features):
    train_target = train_features["is_fraud"]
    test_target = test_features["is_fraud"]
    train_x = train_features.drop(columns=["is_fraud"])
    test_x = test_features.drop(columns=["is_fraud"])

    categorical_columns = train_x.select_dtypes(include=["object", "category"]).columns.tolist()
    print("Encoding (n categories, baseline):",
          {c: (train_x[c].nunique(), train_x[c].value_counts().index[0]) for c in categorical_columns})

    train_parts = [train_x.drop(columns=categorical_columns)]
    test_parts = [test_x.drop(columns=categorical_columns)]
    for column in categorical_columns:
        # Most frequent training category first, so drop_first makes it the baseline
        baseline = train_x[column].value_counts().index[0]
        others = sorted(
            set(pd.concat([train_x[column], test_x[column]]).dropna().unique()) - {baseline}
        )
        categories = pd.Index([baseline] + others)
        dtype = pd.CategoricalDtype(categories=categories)
        train_parts.append(pd.get_dummies(train_x[column].astype(dtype), prefix=column, drop_first=True, sparse=True))
        test_parts.append(pd.get_dummies(test_x[column].astype(dtype), prefix=column, drop_first=True, sparse=True))

    train_encoded = pd.concat(train_parts, axis=1).reset_index(drop=True)
    test_encoded = pd.concat(test_parts, axis=1).reset_index(drop=True)
    train_encoded.insert(0, "is_fraud", train_target.reset_index(drop=True).astype("int8"))
    test_encoded.insert(0, "is_fraud", test_target.reset_index(drop=True).astype("int8"))
    return train_encoded, test_encoded


train_encoded, test_encoded = one_hot_encode(train_features, test_features)

print("train:", train_encoded.shape, "| test:", test_encoded.shape)
print("Same columns:", list(train_encoded.columns) == list(test_encoded.columns))

Encoding (n categories, baseline): {'category': (14, 'gas_transport'), 'gender': (2, 'F'), 'state': (51, 'TX'), 'kyc_employment_status': (4, 'Employed'), 'kyc_residency_status': (3, 'Citizen'), 'kyc_id_verification_method': (4, 'Document (in person)')}
train: (1296675, 111) | test: (555719, 111)
Same columns: True


### Final dataset preview

All columns of the processed training data (`is_fraud` first, then numeric features, then the one-hot indicators).

In [12]:
train_encoded.head()

,is_fraud,amt,city_pop,beh_txn_count_1h,beh_txn_count_24h,beh_log_amt_sum_24h,beh_log_secs_since_prev,beh_amt_to_cust_avg,month,day_of_month,hour,day_of_week,is_weekend,is_night,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,age,log_amt,amt_is_large,log_distance_km,distance_is_large,kyc_log_annual_income,kyc_credit_score,kyc_declared_txn_per_month,kyc_num_products,kyc_account_tenure_years,kyc_address_verified,kyc_phone_verified,kyc_email_verified,kyc_pep_flag,kyc_review_overdue,kyc_sanctions_potential_match,kyc_risk_rating,kyc_amt_to_declared_spend,kyc_amt_to_credit_limit,category_entertainment,category_food_dining,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_M,state_AK,state_AL,state_AR,state_AZ,state_CA,state_CO,state_CT,state_DC,state_DE,state_FL,state_GA,state_HI,state_IA,state_ID,state_IL,state_IN,state_KS,state_KY,state_LA,state_MA,state_MD,state_ME,state_MI,state_MN,state_MO,state_MS,state_MT,state_NC,state_ND,state_NE,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_SD,state_TN,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,kyc_employment_status_Retired,kyc_employment_status_Self-employed,kyc_employment_status_Student,kyc_residency_status_Permanent resident,kyc_residency_status_Visa holder,kyc_id_verification_method_Database check,kyc_id_verification_method_Document + biometric,kyc_id_verification_method_Document upload
0,0,4.97,3495,1.0,1.0,1.786747,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,31.0,1.786747,0,4.376984,0,11.050890,591,41,3,7.498973,1,1,1,0,0,0,0,0.005522,0.000710,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
1,0,107.23,149,1.0,1.0,4.684259,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,41.0,4.684259,0,3.440808,0,11.211820,703,72,1,1.815195,1,1,1,0,0,0,0,0.049874,0.007659,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False
2,0,220.11,4154,1.0,1.0,5.398660,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,57.0,5.398660,1,4.693237,0,11.018629,557,29,5,3.857632,1,1,1,0,0,0,0,0.209629,0.048913,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,0,45.00,1939,1.0,1.0,3.828641,14.767941,1.0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,52.0,3.828641,0,4.571337,0,10.596635,682,33,1,3.411362,1,1,1,0,0,0,0,0.037500,0.004500,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Fa

In [13]:
columns = pd.Series(train_encoded.columns)
groups = {
    "target": columns[columns == "is_fraud"],
    "category dummies": columns[columns.str.startswith("category_")],
    "gender dummies": columns[columns.str.startswith("gender_")],
    "state dummies": columns[columns.str.startswith("state_")],
    "KYC features": columns[columns.str.startswith("kyc_")],
    "behavioural features": columns[columns.str.startswith("beh_")],
}
groups["numeric features"] = columns[~columns.isin(pd.concat(groups.values()))]
for name, cols in groups.items():
    print(f"{name} ({len(cols)}): {', '.join(cols)}\n")

target (1): is_fraud

category dummies (13): category_entertainment, category_food_dining, category_grocery_net, category_grocery_pos, category_health_fitness, category_home, category_kids_pets, category_misc_net, category_misc_pos, category_personal_care, category_shopping_net, category_shopping_pos, category_travel

gender dummies (1): gender_M

state dummies (50): state_AK, state_AL, state_AR, state_AZ, state_CA, state_CO, state_CT, state_DC, state_DE, state_FL, state_GA, state_HI, state_IA, state_ID, state_IL, state_IN, state_KS, state_KY, state_LA, state_MA, state_MD, state_ME, state_MI, state_MN, state_MO, state_MS, state_MT, state_NC, state_ND, state_NE, state_NH, state_NJ, state_NM, state_NV, state_NY, state_OH, state_OK, state_OR, state_PA, state_RI, state_SC, state_SD, state_TN, state_UT, state_VA, state_VT, state_WA, state_WI, state_WV, state_WY

KYC features (22): kyc_log_annual_income, kyc_credit_score, kyc_declared_txn_per_month, kyc_num_products, kyc_account_tenure_years

## 11. Multicollinearity check

Two checks on a 200,000-row sample of the training features (standardised, plus an intercept column):

- **Rank:** full rank means no exact linear dependency remains (no dummy trap, no redundant derived columns).
- **VIF (variance inflation factor):** `1 / (1 − R²)` from regressing each feature on all others. Above ~10 is usually considered problematic.

All VIFs are below 10. The highest are the time features, which is expected because `hour`/`hour_sin`/`hour_cos`/`is_night` all describe the same hour and `day_of_week`/`dow_*`/`is_weekend` the same day. Multicollinearity only matters for interpreting individual logistic-regression coefficients; tree models are unaffected.

For reference, before this cleanup the VIFs reached ~160,000 (`year`/`unix_time`/`month`), ~650 (`lat`/`long`/`merch_lat`/`merch_long`), ~80 (distance interactions) and ~40 (state dummies with a rare baseline).

For the KYC and behavioural features, two first designs were replaced for the same reason: `log(amt) − log(customer average)` was a near-copy of `log_amt` (VIF ~35), so `beh_amt_to_cust_avg` is a plain ratio instead, and the credit limit level (VIF ~12, derived from income) was dropped in favour of the amount-to-limit ratio.

In [14]:
sample = train_encoded.drop(columns=["is_fraud"]).sample(200_000, random_state=0)
sample = sample.apply(
    lambda s: s.sparse.to_dense() if isinstance(s.dtype, pd.SparseDtype) else s
).astype(float)
Z = (sample - sample.mean()) / sample.std()
M = np.column_stack([np.ones(len(Z)), Z.to_numpy()])

print(f"Columns (incl. intercept): {M.shape[1]}, rank: {np.linalg.matrix_rank(M)}")


def vif(Z):
    values = {}
    for column in Z.columns:
        y = Z[column].to_numpy()
        X = np.column_stack([np.ones(len(Z)), Z.drop(columns=column).to_numpy()])
        residual = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
        values[column] = y.var() / residual.var()
    return pd.Series(values).sort_values(ascending=False)


vifs = vif(Z)
vifs.head(15).round(1)

Columns (incl. intercept): 111, rank: 111


hour                         9.6
is_weekend                   9.4
day_of_week                  8.9
is_night                     7.0
kyc_amt_to_credit_limit      5.9
kyc_amt_to_declared_spend    5.3
kyc_risk_rating              4.6
beh_amt_to_cust_avg          4.5
hour_sin                     3.5
amt                          3.3
dow_sin                      3.0
kyc_address_verified         3.0
hour_cos                     3.0
dow_cos                      2.9
log_amt                      2.8
dtype: float64

## 12. Save

In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_encoded.to_csv(OUTPUT_DIR / "fraudTrain_features.csv", index=False)
test_encoded.to_csv(OUTPUT_DIR / "fraudTest_features.csv", index=False)

print(f"Wrote {train_encoded.shape[0]:,} rows x {train_encoded.shape[1]:,} columns (train, incl. is_fraud)")
print(f"Wrote {test_encoded.shape[0]:,} rows x {test_encoded.shape[1]:,} columns (test, incl. is_fraud)")

Wrote 1,296,675 rows x 111 columns (train, incl. is_fraud)
Wrote 555,719 rows x 111 columns (test, incl. is_fraud)
